In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("185").setMaster("local[4]")
spark = SparkSession.builder.config(conf = conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/28 04:00:52 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface enp0s3)
25/08/28 04:00:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/28 04:00:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
Table: Employee

+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| id           | int     |
| name         | varchar |
| salary       | int     |
| departmentId | int     |
+--------------+---------+
id is the primary key (column with unique values) for this table.
departmentId is a foreign key (reference column) of the ID from the Department table.
Each row of this table indicates the ID, name, and salary of an employee. 
It also contains the ID of their department.
 

Table: Department

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| name        | varchar |
+-------------+---------+
id is the primary key (column with unique values) for this table.
Each row of this table indicates the ID of a department and its name.
 

A company's executives are interested in seeing who earns the most money 
in each of the company's departments. 
A high earner in a department is an employee who has a salary in the 
top three unique salaries for that department.

Write a solution to find the employees who are high earners in each of the departments.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Employee table:
+----+-------+--------+--------------+
| id | name  | salary | departmentId |
+----+-------+--------+--------------+
| 1  | Joe   | 85000  | 1            |
| 2  | Henry | 80000  | 2            |
| 3  | Sam   | 60000  | 2            |
| 4  | Max   | 90000  | 1            |
| 5  | Janet | 69000  | 1            |
| 6  | Randy | 85000  | 1            |
| 7  | Will  | 70000  | 1            |
+----+-------+--------+--------------+
Department table:
+----+-------+
| id | name  |
+----+-------+
| 1  | IT    |
| 2  | Sales |
+----+-------+
Output: 
+------------+----------+--------+
| Department | Employee | Salary |
+------------+----------+--------+
| IT         | Max      | 90000  |
| IT         | Joe      | 85000  |
| IT         | Randy    | 85000  |
| IT         | Will     | 70000  |
| Sales      | Henry    | 80000  |
| Sales      | Sam      | 60000  |
+------------+----------+--------+
Explanation: 
In the IT department:
- Max earns the highest unique salary
- Both Randy and Joe earn the second-highest unique salary
- Will earns the third-highest unique salary

In the Sales department:
- Henry earns the highest salary
- Sam earns the second-highest salary
- There is no third-highest salary as there are only two employees
'''

In [2]:
employee_data = [
(1,'Joe'  ,85000,1),
(2,'Henry',80000,2),
(3,'Sam'  ,60000,2),
(4,'Max'  ,90000,1),
(5,'Janet',69000,1),
(6,'Randy',85000,1),
(7,'Will' ,70000,1)
]
employee_schema = ['id','name','salary','departmentId']
department_data = [
(1,'IT'    ),
(2,'Sales' )
]
department_schema = ['id','name']

In [3]:
employee_df = spark.createDataFrame(data=employee_data,schema=employee_schema)
department_df = spark.createDataFrame(data=department_data , schema=department_schema)
employee_df.show()
department_df.show()

+---+-----+------+------------+
| id| name|salary|departmentId|
+---+-----+------+------------+
|  1|  Joe| 85000|           1|
|  2|Henry| 80000|           2|
|  3|  Sam| 60000|           2|
|  4|  Max| 90000|           1|
|  5|Janet| 69000|           1|
|  6|Randy| 85000|           1|
|  7| Will| 70000|           1|
+---+-----+------+------------+

+---+-----+
| id| name|
+---+-----+
|  1|   IT|
|  2|Sales|
+---+-----+



In [15]:
from pyspark.sql.window import Window
WindoeSpec = Window.partitionBy(F.col("departmentId")).orderBy(F.col("salary").desc())

salary_rnk_df = employee_df.select(
                F.col("id"),F.col("name"),F.col("salary"),F.col("departmentId"),
                F.dense_rank().over(WindoeSpec).alias("rnk")
)
salary_rnk_df.alias("t").join(department_df.alias("t1"),
                              F.col("t.departmentId") == F.col("t1.id"),
                              'left'
                             )\
                        .where(F.col("rnk") <= 3)\
                        .orderBy(F.col("t1.name").asc())\
                        .select(F.col("t1.name").alias("Department"),
                                F.col("t.name").alias("Employee"),
                                F.col("t.salary").alias("Salary")
                               )\
                        .show()

+----------+--------+------+
|Department|Employee|Salary|
+----------+--------+------+
|        IT|     Max| 90000|
|        IT|     Joe| 85000|
|        IT|   Randy| 85000|
|        IT|    Will| 70000|
|     Sales|   Henry| 80000|
|     Sales|     Sam| 60000|
+----------+--------+------+



## SQL Solution
<pre>
WITH SALARY_RANK as (
SELECT id,name,salary,departmentId, DENSE_RANK() OVER(PARTITION BY departmentId ORDER BY salary desc) rnk
FROM Employee  
)
SELECT t1.name as  Department, t.name Employee, t.salary 
FROM SALARY_RANK t
LEFT JOIN  Department t1 ON t.departmentId = t1.id
WHERE rnk <= 3 
ORDER BY t1.name
</pre>